In [1]:
## Libraries, modules
import soundfile as sf
import matplotlib.pyplot as plt
import os
import numpy as np
from scipy.signal import square, ShortTimeFFT
from scipy.signal.windows import hann
import cv2
from skimage.feature import peak_local_max

## Recording file
rec_filename = "004_675.269_seq.flac" # "000_1.392_seq.flac" "001_235.717_seq.flac" "002_340.336_seq.flac" "003_454.469_seq.flac" "004_675.269_seq.flac" "005_902.896_seq.flac"

## Read full audio recording
audio_rec, fs = sf.read("./signal_sequences/"+rec_filename)
data = audio_rec[:,0] + audio_rec[:,0] # LSumming the two doesn't improve SNR, but doesn't hurt either.

## Initialize list of detected signals
signals_detected = [
    ["Signal Type", "Time (s)", "Frequency (Hz)", "Correlation Value"]
]

LibsndfileError: Error opening './signal_sequences/004_675.269_seq.flac': System error.

In [ ]:
## Generate Spectrogram (2D FFT)
# Parameters
win_size = 2048  # Window size, 1024 to 4096 are ideal
hop_size = int(win_size*0.5)  # Overlap
fs = 48000  # Your sampling frequency
win = hann(win_size)

# Spectrogram
SFT = ShortTimeFFT(win, hop=hop_size, fs=fs) # Initialize ShortTimeFFT
Sx2 = SFT.spectrogram(data) # Compute the spectrogram (absolute square of STFT)
Sx2[Sx2 == 0] = np.min(Sx2[np.nonzero(Sx2)]) # Substitute zeros with the minimum non-zero value to avoid log(0)
Sx2_dB = 10 * np.log10(Sx2) # Convert spectrogram to dB
t_min, t_max, f_min, f_max = SFT.extent(len(audio_rec), center_bins=True) # Get frequency and time boundaries
f_bins, t_bins = Sx2.shape # Get the number of frequency and time bins

In [ ]:
## Crop spectrogram to avoid high frequency aliased signals
# Frequency range
f_low = 0  # Hz
f_high = 15000  # Hz
f_0 = int(f_low * f_bins / (f_max - f_min))
f_1 = int(f_high * f_bins / (f_max - f_min))

# Time range
t_start = t_min
t_end = t_max
t_0 = np.searchsorted(np.linspace(t_min, t_max, t_bins), t_start)
t_1 = np.searchsorted(np.linspace(t_min, t_max, t_bins), t_end)

# Crop the spectrogram to the selected frequency and time ranges
recording_crop = Sx2_dB[f_0:f_1, t_0:t_1].astype(np.float32)

In [ ]:
## Blank first 9.9 s, the first 10 s shouldn't contain any signal
recording_crop[:, 0:int(fs/2048*2*9.9)] = np.min(recording_crop)

In [ ]:
## EOL signal

# Load EOL signal
EOL_signal = np.genfromtxt('signal_samples/end_signal_48k-2048.csv', delimiter=',').astype(np.float32)

# EOL signal correlation search based on OpenCV match template
correlation_opencv = cv2.matchTemplate(recording_crop, EOL_signal, cv2.TM_CCOEFF_NORMED)

# Find local maxima locations
dist = 1
local_maxima = peak_local_max(correlation_opencv, min_distance=dist, threshold_abs=0.35)

# Sort the local maxima by the second index (time sample)
sorted_indices = np.argsort(local_maxima[:, 1])

# Convert the local maxima indices to time and frequency values
detected_times = np.linspace(t_min, t_max, t_bins)[local_maxima[sorted_indices, 1]]  # Time values from indices
detected_freqs = np.linspace(f_min, f_max, f_bins)[local_maxima[sorted_indices, 0]]  # Frequency values from indices
corr_values = correlation_opencv[local_maxima[sorted_indices, 0], local_maxima[sorted_indices, 1]]

# Add to list of detected signals
for i in range(len(detected_times)):
    signals_detected.append([
        "X",  # Signal type
        round(detected_times[i], 2),  # Time rounded to 2 decimal places
        int(detected_freqs[i]),  # Frequency as an integer
        round(corr_values[i], 2)  # Correlation value rounded to 2 decimal places
    ])

# Print the detected signals
for row in signals_detected:
    print(row)

In [ ]:
## Blank FFT values to the minimum (near 0) where the EOL signal occurs
# Define the width of the end_signal (in terms of time bins)
signal_time_width = EOL_signal.shape[1]  # Time range of end_signal (in time bins)
recording_crop_blank = recording_crop.copy()

# Modify the spectrogram based on local maxima
for local_max in local_maxima:
    t_index = local_max[1]  # Time index where end_signal occurs
    f_index = local_max[0]  # Frequency index where end_signal occurs
    
    # Set a region of the spectrogram around the detected location to the minimum value
    t_start_idx = max(0, t_index - (1520//2-EOL_signal.shape[1]))
    
    # Set **all** frequency bins for the detected time window to the minimum value
    recording_crop_blank[:, t_start_idx:] = np.min(recording_crop) #recording_crop_blank[:, t_start_idx:] - 50

# # Plot the modified spectrogram
# plt.figure(figsize=(15, 6))
# plt.pcolormesh(np.linspace(t_min, t_max, t_bins)[t_0:t_1], np.linspace(f_min, f_max, f_bins)[f_0:f_1], recording_crop_blank)
# plt.colorbar()
# plt.ylabel('Frequency [Hz]')
# plt.xlabel('Time [sec]')
# plt.show()

In [ ]:
## G signal

# Load G signal
G_signal = np.genfromtxt('signal_samples/G_signal_48k-2048.csv', delimiter=',').astype(np.float32)

## Search based on OpenCV match template, very fast
#recording_crop_blank = recording_crop_blank.astype(np.float32)
correlation_opencv = cv2.matchTemplate(recording_crop_blank, G_signal, cv2.TM_CCOEFF_NORMED)

# Find local maxima locations
dist = 1
local_maxima = peak_local_max(correlation_opencv, min_distance=dist, threshold_abs=0.35)

# Sort the local maxima by the second index (time sample)
sorted_indices = np.argsort(local_maxima[:, 1])

# Convert the local maxima indices to time and frequency values
detected_times = np.linspace(t_min, t_max, t_bins)[local_maxima[sorted_indices, 1]]  # Time values from indices
detected_freqs = np.linspace(f_min, f_max, f_bins)[local_maxima[sorted_indices, 0]]  # Frequency values from indices
corr_values = correlation_opencv[local_maxima[sorted_indices, 0], local_maxima[sorted_indices, 1]]

# Add to list of detected signals
for i in range(len(detected_times)):
    signals_detected.append([
        "G",  # Signal type
        round(detected_times[i], 2),  # Time rounded to 2 decimal places
        int(detected_freqs[i]),  # Frequency as an integer
        round(corr_values[i], 2)  # Correlation value rounded to 2 decimal places
    ])

# Print the detected signals
for row in signals_detected:
    print(row)

In [ ]:
## Blank FFT values to the minimum (near 0) where the G signal occurs
signal_time_width = G_signal.shape[1]  # Time range of end_signal (in time bins)
recording_crop_blank_G = recording_crop_blank.copy()

# Modify the spectrogram based on local maxima
for local_max in local_maxima:
    t_index = local_max[1]  # Time index where end_signal occurs
    f_index = local_max[0]  # Frequency index where end_signal occurs
    
    # Set a region of the spectrogram around the detected location to the minimum value
    t_start_idx = max(0, t_index)
    t_end_idx = min(t_bins, t_index + signal_time_width)
    
    # Set **all** frequency bins for the detected time window to the minimum value
    recording_crop_blank_G[:, t_start_idx:t_end_idx] = np.min(recording_crop)

# # Plot the modified spectrogram
# plt.figure(figsize=(15, 6))
# plt.pcolormesh(np.linspace(t_min, t_max, t_bins)[t_0:t_1], np.linspace(f_min, f_max, f_bins)[f_0:f_1], recording_crop_blank_G)
# plt.colorbar()
# plt.ylabel('Frequency [Hz]')
# plt.xlabel('Time [sec]')
# plt.show()

In [ ]:
## F signal

# Load F signal
F_signal = np.genfromtxt('signal_samples/F_signal_48k-2048.csv', delimiter=',').astype(np.float32)

# Search based on OpenCV match template, very fast
#recording_crop_blank_G = recording_crop_blank_G.astype(np.float32)
correlation_opencv = cv2.matchTemplate(recording_crop_blank_G, F_signal, cv2.TM_CCOEFF_NORMED)

## Find and print local maxima locations
dist = 1 #correlation_opencv.shape[0]//4
local_maxima = peak_local_max(correlation_opencv, min_distance=dist, threshold_abs=0.35)

# Sort the local maxima by the second index (time sample)
sorted_indices = np.argsort(local_maxima[:, 1])

# Convert the local maxima indices to time and frequency values
detected_times = np.linspace(t_min, t_max, t_bins)[local_maxima[sorted_indices, 1]]  # Time values from indices
detected_freqs = np.linspace(f_min, f_max, f_bins)[local_maxima[sorted_indices, 0]]  # Frequency values from indices
corr_values = correlation_opencv[local_maxima[sorted_indices, 0], local_maxima[sorted_indices, 1]]

# Add to list of detected signals
for i in range(len(detected_times)):
    signals_detected.append([
        "F",  # Signal type
        round(detected_times[i], 2),  # Time rounded to 2 decimal places
        int(detected_freqs[i]),  # Frequency as an integer
        round(corr_values[i], 2)  # Correlation value rounded to 2 decimal places
    ])

# Print the detected signals
for row in signals_detected:
    print(row)

In [ ]:
## Blank FFT values to the minimum (near 0) where the F signal occurs
# Define the width of the end_signal (in terms of time bins)
signal_time_width = F_signal.shape[1]  # Time range of end_signal (in time bins)
recording_crop_blank_F = recording_crop_blank_G.copy()

# Modify the spectrogram based on local maxima
for local_max in local_maxima:
    t_index = local_max[1]  # Time index where end_signal occurs
    f_index = local_max[0]  # Frequency index where end_signal occurs
    
    # Set a region of the spectrogram around the detected location to the minimum value
    t_start_idx = max(0, t_index)
    t_end_idx = min(t_bins, t_index + signal_time_width)
    
    # Set **all** frequency bins for the detected time window to the minimum value
    recording_crop_blank_F[:, t_start_idx:t_end_idx] = np.min(recording_crop)

# # Plot the modified spectrogram
# plt.figure(figsize=(15, 6))
# plt.pcolormesh(np.linspace(t_min, t_max, t_bins)[t_0:t_1], np.linspace(f_min, f_max, f_bins)[f_0:f_1], recording_crop_blank_F)
# plt.colorbar()
# plt.ylabel('Frequency [Hz]')
# plt.xlabel('Time [sec]')
# plt.show()

In [ ]:
## E signal

# Load E signal
E_signal = np.genfromtxt('signal_samples/E_signal_48k-2048.csv', delimiter=',').astype(np.float32)

# Search based on OpenCV match template, very fast
#recording_crop = recording_crop.astype(np.float32)
correlation_opencv = cv2.matchTemplate(recording_crop_blank_F, E_signal, cv2.TM_CCOEFF_NORMED)

## Find and print local maxima locations
dist = 1 #correlation_opencv.shape[0]//4
local_maxima = peak_local_max(correlation_opencv, min_distance=dist, threshold_abs=0.35)

# Sort the local maxima by the second index (time sample)
sorted_indices = np.argsort(local_maxima[:, 1])

# Convert the local maxima indices to time and frequency values
detected_times = np.linspace(t_min, t_max, t_bins)[local_maxima[sorted_indices, 1]]  # Time values from indices
detected_freqs = np.linspace(f_min, f_max, f_bins)[local_maxima[sorted_indices, 0]]  # Frequency values from indices
corr_values = correlation_opencv[local_maxima[sorted_indices, 0], local_maxima[sorted_indices, 1]]

# Add to list of detected signals
for i in range(len(detected_times)):
    signals_detected.append([
        "E",  # Signal type
        round(detected_times[i], 2),  # Time rounded to 2 decimal places
        int(detected_freqs[i]),  # Frequency as an integer
        round(corr_values[i], 2)  # Correlation value rounded to 2 decimal places
    ])

# Print the detected signals
for row in signals_detected:
    print(row)

In [ ]:
## Blank FFT values to the minimum (near 0) where the E occurs
# Define the width of the end_signal (in terms of time bins)
signal_time_width = E_signal.shape[1]  # Time range of end_signal (in time bins)
recording_crop_blank_E = recording_crop_blank_F.copy()

# Modify the spectrogram based on local maxima
for local_max in local_maxima:
    t_index = local_max[1]  # Time index where end_signal occurs
    f_index = local_max[0]  # Frequency index where end_signal occurs
    
    # Set a region of the spectrogram around the detected location to the minimum value
    t_start_idx = max(0, t_index - (59-E_signal.shape[1]))
    t_end_idx = min(t_bins, t_index + signal_time_width)
    
    # Set **all** frequency bins for the detected time window to the minimum value
    recording_crop_blank_E[:, t_start_idx:t_end_idx] = np.min(recording_crop)

# # Plot the modified spectrogram
# plt.figure(figsize=(15, 6))
# plt.pcolormesh(np.linspace(t_min, t_max, t_bins)[t_0:t_1], np.linspace(f_min, f_max, f_bins)[f_0:f_1], recording_crop_blank_E)
# plt.colorbar()
# plt.ylabel('Frequency [Hz]')
# plt.xlabel('Time [sec]')
# plt.show()

In [ ]:
## D signal

# Load bat-signal
D_signal = np.genfromtxt('signal_samples/D_signal_48k-2048.csv', delimiter=',').astype(np.float32)

# Search based on OpenCV match template, very fast
#recording_crop_blank_E = recording_crop_blank_E.astype(np.float32)
correlation_opencv = cv2.matchTemplate(recording_crop_blank_E, D_signal, cv2.TM_CCOEFF_NORMED)

## Find and print local maxima locations
dist = 1 #correlation_opencv.shape[0]//4
local_maxima = peak_local_max(correlation_opencv, min_distance=dist, threshold_abs=0.3)

# Sort the local maxima by the second index (time sample)
sorted_indices = np.argsort(local_maxima[:, 1])

# Convert the local maxima indices to time and frequency values
detected_times = np.linspace(t_min, t_max, t_bins)[local_maxima[sorted_indices, 1]]  # Time values from indices
detected_freqs = np.linspace(f_min, f_max, f_bins)[local_maxima[sorted_indices, 0]]  # Frequency values from indices
corr_values = correlation_opencv[local_maxima[sorted_indices, 0], local_maxima[sorted_indices, 1]]

# Add to list of detected signals
for i in range(len(detected_times)):
    signals_detected.append([
        "D",  # Signal type
        round(detected_times[i], 2),  # Time rounded to 2 decimal places
        int(detected_freqs[i]),  # Frequency as an integer
        round(corr_values[i], 2)  # Correlation value rounded to 2 decimal places
    ])

# Print the detected signals
for row in signals_detected:
    print(row)

In [ ]:
## Blank FFT values to the minimum (near 0) where the D signal occurs
# Define the width of the end_signal (in terms of time bins)
signal_time_width = D_signal.shape[1]  # Time range of end_signal (in time bins)
recording_crop_blank_D = recording_crop_blank_E.copy()

# Modify the spectrogram based on local maxima
for local_max in local_maxima:
    t_index = local_max[1]  # Time index where end_signal occurs
    f_index = local_max[0]  # Frequency index where end_signal occurs
    
    # Set a region of the spectrogram around the detected location to the minimum value
    t_start_idx = max(0, t_index)
    t_end_idx = min(t_bins, t_index + 75)
    
    # Set **all** frequency bins for the detected time window to the minimum value
    recording_crop_blank_D[:, t_start_idx:t_end_idx] = np.min(recording_crop)

# # Plot the modified spectrogram
# plt.figure(figsize=(15, 6))
# plt.pcolormesh(np.linspace(t_min, t_max, t_bins)[t_0:t_1], np.linspace(f_min, f_max, f_bins)[f_0:f_1], recording_crop_blank_D)
# plt.colorbar()
# plt.ylabel('Frequency [Hz]')
# plt.xlabel('Time [sec]')
# plt.show()

In [ ]:
## Measure background noise level
# Spectrogram bandpass filter
def spec_bandpass(spec, f_max, f_bins, f_low, f_high):
    f_bl = int(f_low/f_max*f_bins) # Low frequency bin
    f_bh = int(f_high/f_max*f_bins) # High frequency bin
    return spec[f_bl:f_bh,:]

# Extract noise band    
noise = spec_bandpass(Sx2_dB, f_max, f_bins, 6300, 6700)

# # Plot 'cropped' spectrogram  (in dB)
# plt.figure(figsize=(15,6))
# plt.imshow(noise, origin='lower', aspect='auto', extent=[t_min, t_max, 6300, 6700])
# plt.colorbar()
# plt.ylabel('Frequency [Hz]')
# plt.xlabel('Time [sec]')
# plt.show()

noise_mean = np.mean(noise)
noise_std = np.std(noise)
#print("mean " + str(noise_mean))
#print("std " + str(noise_std))

In [ ]:
## Measure signal level in strong "tone" band
f_low = 6600
f_high = 7700
sig_bandpass = spec_bandpass(recording_crop_blank_D, f_max, f_bins, f_low, f_high)

# Plot 'cropped' spectrogram  (in dB)
plt.figure(figsize=(15,6))
plt.imshow(sig_bandpass, origin='lower', aspect='auto', extent=[t_min, t_max, f_low, f_high])
plt.colorbar()
plt.ylabel('Frequency [Hz]')
plt.xlabel('Time [sec]')
plt.show()

sig_level = np.max(sig_bandpass, axis=0)

plt.figure(figsize=(15,6))
plt.plot(sig_level)
plt.axhline(y = noise_mean + noise_std*2.25, color = 'r', linestyle = '-')
plt.ylim(-30, 5)
plt.show()

In [ ]:
## Create "detection" array
threshold = noise_mean + noise_std*2.25 # Threshold level to detect a signal
above_threshold = (sig_level > threshold).astype(int) # Array of detections (1) and non-detections (0)

# Function to remove short detections
def remove_short_ones(arr, min_ones):
    result = arr.copy()
    i = 0
    while i < len(result):
        if result[i] == 1:  # Found the start of a sequence of 1s
            start = i
            while i < len(result) and result[i] == 1:  # Find the end of this sequence
                i += 1
            end = i
            if end - start < min_ones:  # If the length of the sequence is shorter than min_ones, set to 0
                for j in range(start, end):
                    result[j] = 0
        else:
            i += 1    
    return result

min_ones = int(fs/win_size*2*0.2) # 0.2 seconds, the shortest portion of signal C is 0.38 s long
sig_detected = remove_short_ones(above_threshold, min_ones)
plt.figure(figsize=(15,6))
plt.plot(above_threshold)
plt.plot(sig_detected)
plt.show()

In [ ]:
## Blanking of noise sections based on previous detections, then bandpass to low frequency strong tone.

# Cropping based on sig_detected, applied to the original cropped spectrogram
recording_crop_blank_tones = recording_crop.copy()
recording_crop_blank_tones[:, np.where(sig_detected == 0)[0]] = np.min(recording_crop)

In [ ]:
# First bandpass filter (wide)
f_low_1 = 6500
f_high_1 = 7200
recording_crop_blank_tones_bp1 = spec_bandpass(recording_crop_blank_tones, f_max, f_bins, f_low_1, f_high_1)

# Plot 'cropped' spectrogram  (in dB)
plt.figure(figsize=(15,6))
plt.imshow(recording_crop_blank_tones_bp1, origin='lower', aspect='auto', extent=[t_min, t_max, f_low_1, f_high_1])
plt.colorbar()
plt.ylabel('Frequency [Hz]')
plt.xlabel('Time [sec]')
plt.show()

In [ ]:
## Find local maxima for each tone (frequency sample), 1st pass
# The frequency resolution is not great, but I don't want to degrade the time resolution.

# Local maxima, this is often noisy though
tones_local_max = np.argmax(recording_crop_blank_tones_bp1, axis=0)

# Find indices nd tone segments
det_idx = np.where(sig_detected == 1)[0] # Find the indices where a tone is s detected
tone_segm_boundaries = np.where(np.diff(det_idx) > 1)[0] + 1 # Find the boundaries where the difference is greater than 1
tone_segm_boundaries = np.concatenate(([0], tone_segm_boundaries, [len(det_idx)])) # Add the first and last index to define the tone segments

# Calculate the median of tones_local_max for each tone segment
medians = []
for i in range(len(tone_segm_boundaries) - 1):
    segment = det_idx[tone_segm_boundaries[i]:tone_segm_boundaries[i+1]]
    medians.append(np.median(tones_local_max[segment]))

# Initialize an array for tones_local_max_med, add the corresponding median for each segment (this works as a filter)
tones_freq_med = np.copy(tones_local_max)
segment_start = 0
for i in range(len(tone_segm_boundaries) - 1):
    segment = det_idx[tone_segm_boundaries[i]:tone_segm_boundaries[i+1]]
    median_value = medians[i]
    tones_freq_med[segment] = median_value

# Plot both original and modified (median) tones
plt.figure(figsize=(15,6))
plt.plot(tones_local_max)
plt.plot(tones_freq_med)
plt.tight_layout()
plt.show()

In [ ]:
# Determine min/max tone frequencies, then run second bandpass (narrow)
margin = 75 # Frequency margin below and above tones
f_low_2 = np.min(tones_freq_med[tones_freq_med != 0])*fs/win_size + f_low_1 - margin
f_high_2 = np.max(tones_freq_med)*fs/win_size + f_low_1 + margin
recording_crop_blank_tones_bp2 = spec_bandpass(recording_crop_blank_tones, f_max, f_bins, f_low_2, f_high_2)

# Plot 'cropped' spectrogram  (in dB)
plt.figure(figsize=(15,6))
plt.imshow(recording_crop_blank_tones_bp2, origin='lower', aspect='auto', extent=[t_min, t_max, f_low_2, f_high_2])
plt.colorbar()
plt.ylabel('Frequency [Hz]')
plt.xlabel('Time [sec]')
plt.show()

In [ ]:
## Find local maxima for each tone (frequency sample), 2nd pass
# The frequency resolution is not great, but I don't want to degrade the time resolution.

# Local maxima, this is often noisy though
tones_local_max = np.argmax(recording_crop_blank_tones_bp2, axis=0)

# Find indices nd tone segments
det_idx = np.where(sig_detected == 1)[0] # Find the indices where a tone is s detected
tone_segm_boundaries = np.where(np.diff(det_idx) > 1)[0] + 1 # Find the boundaries where the difference is greater than 1
tone_segm_boundaries = np.concatenate(([0], tone_segm_boundaries, [len(det_idx)])) # Add the first and last index to define the tone segments

# Calculate the median of tones_local_max for each tone segment
medians = []
for i in range(len(tone_segm_boundaries) - 1):
    segment = det_idx[tone_segm_boundaries[i]:tone_segm_boundaries[i+1]]
    medians.append(np.median(tones_local_max[segment]))

# Initialize an array for tones_local_max_med, add the corresponding median for each segment (this works as a filter)
tones_freq_med = np.copy(tones_local_max)
segment_start = 0
for i in range(len(tone_segm_boundaries) - 1):
    segment = det_idx[tone_segm_boundaries[i]:tone_segm_boundaries[i+1]]
    median_value = medians[i]
    tones_freq_med[segment] = median_value

# Plot both original and modified (median) tones
plt.figure(figsize=(15,6))
plt.plot(tones_local_max)
plt.plot(tones_freq_med)
plt.tight_layout()
plt.show()

In [ ]:
## Find tone signals (as sequences of values above 0)
def find_tone_signals(bin_list, tones_freq_med):
    tone_signals = []
    start_idx = None
    for i, num in enumerate(bin_list):  # Go through detection array
        if num > 0:
            if start_idx is None:  # Beginning of a new sequence of 1s
                start_idx = i
        elif num == 0 and start_idx is not None:  # End the sequence when we hit a 0 after detecting 1s
            tone_signals.append((start_idx, i - 1, tones_freq_med[start_idx]))  # Add median value as third column
            start_idx = None    
    if start_idx is not None:  # If we end with a sequence of 1s and it's still open, close it
        tone_signals.append((start_idx, len(bin_list) - 1, tones_freq_med[start_idx]))  # Add median value as third column
    return tone_signals

tone_signals = find_tone_signals(sig_detected, tones_freq_med)

# Print the result with start_idx, end_idx, and median value
for row in tone_signals:
    print(str(row) + " " + str(row[1]-row[0]))

In [ ]:
## Detect which tones occur
# This is currently hard-coded for a 2048 FFT with 50% overlap
def detect_tones(tone_signals):
    tones_list = [] # List with final output
    i = 0 # Initialize row index
    while i < len(tone_signals):
        duration_samples = tone_signals[i][1] - tone_signals[i][0] # Duration of each sequence, in samples
        row = [tone_signals[i][0], tone_signals[i][1], duration_samples, "", ""] # Add empty 5th column for frequency
        if (duration_samples > 150): # Tone A is usually ~141 samples, so longer tones mean that A & B or A & C are too close to each other
            raise ValueError("Check for possible A+B or A+C.")
        elif (duration_samples > 130): # Tone A is usually ~141 samples
            row[3] = "A"
            row[4] = tone_signals[i][2] # Frequency sample
        elif (duration_samples > 90): # Tone B is usually ~83 samples, so longer tones mean that B & C are too close to each other
            raise ValueError("Check for possible B+C.")
        elif (duration_samples > 70): # Tone B is usually > 80 samples
            row[3] = "B"
            row[4] = tone_signals[i][2] # Frequency sample
        elif (duration_samples > 25): # Tone C is made of two tones, the first is usually > 33 samples
            if i + 1 < len(tone_signals): # Check if there is a next row
                next_duration_samples = tone_signals[i + 1][1] - tone_signals[i + 1][0]
                if (next_duration_samples < 20):  # The second tone is usually < 17 samples
                    tones_list.append([tone_signals[i][0], tone_signals[i + 1][1], (tone_signals[i + 1][1] - tone_signals[i][0]), "C", tone_signals[i][2]])  # Merge rows and add "C" label
                    i += 2 # Skip the next row as it was merged with the current one
                    continue # Skip current row because it's merged with the next row
                else:
                    raise ValueError("C tone only partially detected (missing 2nd half).")
            else:
                raise ValueError("C tone only partially detected (end of tone_signals).")
        else:
            raise ValueError("C tone only partially detected (missing 1st half).")
        
        tones_list.append(row)
        i += 1 # Move to the next row

    return tones_list

# Process the tone sequences
try:
    tones_list = detect_tones(tone_signals)
    for row in tones_list:
        print(row)
except ValueError as e:
    print(f"Error: {e}")

In [ ]:
# Convert to time and frequency values
tone_type = np.array(tones_list)[:, 3]
tone_times = np.array([row[0] for row in tones_list])
tone_freqs = np.array([row[4] for row in tones_list])
detected_times = np.linspace(t_min, t_max, t_bins)[tone_times] # Time values from indices
detected_freqs = np.linspace(f_min, f_max, f_bins)[tone_freqs] + f_low_2  # Frequency values from indices, corrected for previous bandpass

# Calculate length metric
length_metric = np.zeros(len(tones_list))
length_samples = np.array([row[2] for row in tones_list]) # Length of tones (in # samples), it shouldn't differ much from 'standard' values
for t in range(len(length_samples)):
    if tone_type[t] == "A":
        length_metric[t] = (1 - (np.abs(length_samples[t] - 141) / 141))  # Tone A
    if tone_type[t] == "B":
        length_metric[t] = (1 - (np.abs(length_samples[t] - 83) / 83))  # Tone B
    if tone_type[t] == "C":
        length_metric[t] = (1 - (np.abs(length_samples[t] - 75) / 75))  # Tone C

# Add to list of detected signals
for i in range(len(detected_times)):
    signals_detected.append([
        tone_type[i], # Signal type
        round(detected_times[i], 2), # Time rounded to 2 decimal places
        int(detected_freqs[i]), # Frequency as an integer
        round(length_metric[i], 2) # Length metric as an integer
    ])

# Sort list based on time
signals_detected = [signals_detected[0]] + sorted(signals_detected[1:], key=lambda x: x[1])

# Print the detected signals
for row in signals_detected:
    print(row)